# Chapter 15 — Queuing Theory Basics
*Track 4: Engineers*

## 🎯 Learning Objectives
- Understand the M/M/1 queue and Little's Law
- Compute average queue length, wait time, and utilisation
- Model real engineering bottlenecks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — The M/M/1 Queue

**Kendall notation:** M/M/1
- First M: Markovian (Poisson) arrivals with rate $\lambda$
- Second M: Markovian (Exponential) service with rate $\mu$
- 1: single server

**Steady-state results (valid when ρ = λ/μ < 1):**

| Metric | Formula |
|--------|---------|
| Utilisation | $\rho = \lambda/\mu$ |
| Avg. customers in system | $L = \rho/(1-\rho)$ |
| Avg. customers in queue | $L_q = \rho^2/(1-\rho)$ |
| Avg. time in system | $W = 1/(\mu - \lambda)$ |
| Avg. wait in queue | $W_q = \lambda/[\mu(\mu-\lambda)]$ |

**Little's Law** (works for any stable queue): $L = \lambda W$

In [ ]:
# M/M/1 metrics as a function of utilisation
rho_values = np.linspace(0.01, 0.99, 200)

L  = rho_values / (1 - rho_values)       # avg customers in system
Lq = rho_values**2 / (1 - rho_values)    # avg queue length

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(rho_values, L,  lw=2, label="L (in system)")
axes[0].plot(rho_values, Lq, lw=2, label="Lq (in queue)")
axes[0].axvline(0.8, color="red", linestyle="--", label="ρ=0.8")
axes[0].set_xlabel("Utilisation ρ"); axes[0].set_ylabel("Average customers")
axes[0].set_ylim(0, 20); axes[0].legend()
axes[0].set_title("M/M/1: Average Queue vs Utilisation")

# Service time = 1/mu = 1, vary lambda
mu = 1.0
lambdas = np.linspace(0.01, 0.99, 200)
W = 1 / (mu - lambdas)
axes[1].plot(lambdas, W, lw=2)
axes[1].set_xlabel("Arrival rate λ"); axes[1].set_ylabel("Avg. time in system W")
axes[1].set_ylim(0, 30); axes[1].set_title("M/M/1: Sojourn Time vs Arrival Rate (μ=1)")
plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Deriving L from Balance Equations

For M/M/1 with $\rho < 1$, steady-state probabilities:
$$\pi_n = (1-\rho)\rho^n$$

Then:
$$L = \sum_{n=0}^\infty n \pi_n = (1-\rho)\rho \frac{d}{d\rho}\left(\sum_{n=0}^\infty \rho^n\right) = \frac{\rho}{1-\rho}$$

In [ ]:
# Verify analytically
rho = 0.7
pi_n = [(1-rho)*rho**n for n in range(1000)]
L_direct = sum(n*p for n, p in enumerate(pi_n))
L_formula = rho / (1 - rho)
print(f"L from summation: {L_direct:.4f}")
print(f"L from formula:   {L_formula:.4f}")

In [ ]:
# Discrete-event simulation of M/M/1 queue
def simulate_mm1(lam, mu, n_customers=5000):
    arrivals = np.cumsum(rng.exponential(1/lam, n_customers))
    services = rng.exponential(1/mu, n_customers)
    start = np.zeros(n_customers)
    finish = np.zeros(n_customers)
    for i in range(n_customers):
        start[i] = max(arrivals[i], finish[i-1]) if i > 0 else arrivals[i]
        finish[i] = start[i] + services[i]
    wait = start - arrivals
    sojourn = finish - arrivals
    return wait.mean(), sojourn.mean()

lam, mu = 0.7, 1.0
rho = lam/mu
w_q_sim, w_sim = simulate_mm1(lam, mu)
print(f"ρ={rho:.1f}")
print(f"Simulated Wq={w_q_sim:.3f}, theory={lam/(mu*(mu-lam)):.3f}")
print(f"Simulated W={w_sim:.3f},  theory={1/(mu-lam):.3f}")

In [ ]:
# Real-world: web server capacity planning
def capacity_plan(lam_peak, mu, sla_wait):
    # Find min servers needed to meet SLA (avg wait < sla_wait).
    for c in range(1, 20):
        rho_c = lam_peak / (c * mu)
        if rho_c >= 1:
            continue
        # M/M/c Erlang-C approximation
        # P0 calculation
        erlang_c_num = (c*rho_c)**c / (np.math.factorial(c) * (1 - rho_c))
        denom = sum((c*rho_c)**k / np.math.factorial(k) for k in range(c)) + erlang_c_num
        P0 = 1 / denom
        Pc = erlang_c_num * P0
        Wq = Pc / (c * mu * (1 - rho_c))
        if Wq < sla_wait:
            return c, rho_c, Wq
    return None

import math
result = capacity_plan(lam_peak=8, mu=3, sla_wait=0.1)
if result:
    c, rho, wq = result
    print(f"Need {c} servers: ρ={rho:.2f}, avg wait={wq:.4f}s (SLA<0.1s)")